# T2 — Matching de Produtos
**Aprendizado de Máquina — Prof. Me. Otávio Parraga**


Dado um texto de consulta heterogêneo (ex.: `FANTA LARANJA 2L C/6`), o sistema encontra o
produto correspondente no catálogo normalizado (`Dados/catalog.csv`, 14.206 produtos).

Duas abordagens são implementadas e comparadas:

| Abordagem | Técnica |
|---|---|
| **1 — NLP Clássico** | TF-IDF (palavras e n-gramas de caracteres) com cosseno + **BM25 implementado do zero** |
| **2 — Deep Learning** | Embeddings semânticos (`paraphrase-multilingual-MiniLM-L12-v2`, local/CPU), variantes densa e híbrida (BM25 top-50 → re-rank neural) |

Métricas: **P@1**, **MRR@5** e **R@5**, calculadas em `queries_val.csv` (desenvolvimento) e,
uma única vez ao final, em `queries_test.csv` (definitivas).

> Este notebook é autocontido (não depende de `src/`), mas o código é o mesmo dos scripts
> em `src/`, que continuam sendo a forma recomendada de reprodução em lote.

## 0. Setup

Imports e caminhos. **Atenção:** nesta máquina o `torch` precisa ser importado **antes** do `pandas` (conflito de DLL no Windows).

In [1]:
import torch  # importar ANTES de pandas (conflito de DLL c10.dll no Windows)

import hashlib, json, math, re, time, unicodedata
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize as l2_normalize

ROOT = Path.cwd()                 # executar o notebook a partir da raiz do projeto
DADOS, CACHE, RESULTS = ROOT / "Dados", ROOT / "cache", ROOT / "results"
RESULTS.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", 90)
print("torch", torch.__version__, "| pandas", pd.__version__)

torch 2.11.0+cpu | pandas 1.5.3


## 1. Exploração dos dados

Carregamos os quatro CSVs **sempre com `dtype=str`** — os `product_id` são códigos EAN com
zero à esquerda (`07891...`) e seriam corrompidos se lidos como inteiros.

In [2]:
catalog = pd.read_csv(DADOS / "catalog.csv", dtype=str)
queries = pd.read_csv(DADOS / "queries.csv", dtype=str)
val     = pd.read_csv(DADOS / "queries_val.csv", dtype=str)
test    = pd.read_csv(DADOS / "queries_test.csv", dtype=str)

print(f"catalog={len(catalog)}  queries={len(queries)}  val={len(val)}  test={len(test)}")
catalog.head(3)

catalog=14206  queries=16441  val=250  test=250


,product_id,product_name,brand_name
0,07891991014915,Refrigerante guaraná antarctica 200ml,Antarctica
1,27897123881039,Água mineral sem gás prata 300ml,Prata
2,67896100501845,Suco de uva orgânico aliança 1 litro,Aliança


In [3]:
# Catálogo: marcas, "categorias" (1ª palavra) e nomes duplicados (ambiguidade!)
dup = catalog[catalog.duplicated("product_name", keep=False)]
print(f"Marcas únicas: {catalog['brand_name'].nunique()}")
print(f"Nomes duplicados sob ids diferentes: {dup['product_name'].nunique()} nomes / {len(dup)} linhas")
display(catalog["product_name"].str.split().str[0].str.lower().value_counts().head(10).to_frame("qtde"))
dup.sort_values("product_name").head(6)

Marcas únicas: 1833
Nomes duplicados sob ids diferentes: 29 nomes / 59 linhas


,qtde
conjunto,347
biscoito,338
chocolate,330
iogurte,281
queijo,261
massa,221
sabonete,218
creme,209
pão,204
cerveja,197


,product_id,product_name,brand_name
10669,07896001250628,Adoçante líquido sucralose linea 75ml,Linea
12084,07896001250611,Adoçante líquido sucralose linea 75ml,Linea
13138,07898994939726,Bebida de aveia orgânica sem glúten nude 1 litro,Nude
6416,07898994939764,Bebida de aveia orgânica sem glúten nude 1 litro,Nude
11670,02079620000002,Bife de contrafilé bovino novilho jovem zaffari,Zaffari
11900,02112110000004,Bife de contrafilé bovino novilho jovem zaffari,Açougue


In [4]:
# Queries: comprimento e tokens de embalagem/quantidade (C/6, 12X..., UND)
tok = queries["text"].str.split().str.len()
print(f"Tokens por query: mediana={tok.median():.0f}  min={tok.min()}  max={tok.max()}")
pack = re.findall(r"\b(?:C/\d+|\d+X\d+\w*|CX\d*|FD\d*|PCT|UND?|UNID)\b",
                  " ".join(queries["text"]).upper())
print(f"Tokens de embalagem: {len(pack)} ocorrências — {Counter(pack).most_common(8)}")

# Cobertura de val/test no catálogo (há NO_MATCH rotulado?)
ids = set(catalog["product_id"])
print(f"val com id no catálogo : {val['matched_id'].isin(ids).sum()}/{len(val)}")
print(f"test com id no catálogo: {test['matched_id'].isin(ids).sum()}/{len(test)}")

Tokens por query: mediana=7  min=1  max=28
Tokens de embalagem: 1064 ocorrências — [('UNID', 240), ('C/12', 176), ('C/6', 173), ('UND', 92), ('C/24', 66), ('CX', 64), ('FD', 35), ('C/8', 25)]
val com id no catálogo : 250/250


test com id no catálogo: 250/250


**Observações da exploração:**
- val/test estão 100% cobertos pelo catálogo → não há NO_MATCH rotulado (tratamos NO_MATCH na Seção 7);
- o estilo de val/test é descritivo (`"Sabonete Líquido Buquê de Jasmim Lux..."`), enquanto
  `queries.csv` é abreviado e ruidoso (`"FINI MINI BANANAS 12X15 GR Und"`);
- `queries.csv` contém famílias inteiras de NO_MATCH: as marcas **Qualitá** (1.234 queries),
  **Taeq**, **Swift**, **Carnilove** e **Mormaii** não existem no catálogo — e não há
  "Cerveja skol lata 350ml" neste catálogo (só Skol Beats);
- o catálogo tem 29 nomes duplicados sob ids distintos → ambiguidade irredutível por texto.

## 2. Pré-processamento textual (Etapa 1.1)

Normalização aplicada **tanto às queries quanto ao catálogo**. Decisões principais:
- `12X350ML` → `350ml` (mantém o tamanho unitário, descarta a multiplicidade);
- `2L`, `2 litros` → `2l` (forma canônica de unidades);
- **mantemos "sem"** (não como stopword): é discriminativo em "sem açúcar/gás/lactose";
- dicionário de abreviações **conservador**: `COND` (condensado×condicionador) e `SAB`
  (sabão×sabonete) ficam de fora de propósito — expansão errada induz match errado.

In [5]:
# -*- coding: utf-8 -*-
"""
Etapa 1.1 — Pré-processamento textual.

Normaliza tanto as queries quanto os nomes de produto do catálogo:
  - minúsculas e remoção de acentos;
  - expansão de "s/" -> "sem" e "c/" -> "com" (quando seguidos de palavra);
  - remoção de quantidades de embalagem (C/6, C/12, CX, FD, 12X350ML -> 350ml, UND...);
  - normalização de unidades de medida (2L -> 2l, 350 ML -> 350ml, 15 GR -> 15g);
  - normalização de abreviações comuns do varejo (cerv -> cerveja, refrig -> refrigerante...);
  - remoção de pontuação e stopwords do português.

Observação metodológica: NÃO removemos a palavra "sem" como stopword, pois ela é
discriminativa em produtos ("sem açúcar", "sem gás", "sem lactose"). "com" é
removida porque a ausência dela não gera confusão ("água gás" ≠ "água sem gás").
"""
import re
import unicodedata

# stopwords do português relevantes para descrições de produto
STOPWORDS = {
    "de", "do", "da", "dos", "das", "e", "o", "a", "os", "as",
    "um", "uma", "para", "por", "em", "no", "na", "nos", "nas", "ao", "com",
}

# abreviações comuns em pedidos de compra -> forma do catálogo.
# Mantemos apenas abreviações NÃO ambíguas: "cond" (condensado x condicionador)
# e "sab" (sabão x sabonete), por exemplo, ficaram de fora de propósito.
ABBREVIATIONS = {
    "cerv": "cerveja",
    "refri": "refrigerante",
    "refrig": "refrigerante",
    "choc": "chocolate",
    "achoc": "achocolatado",
    "bisc": "biscoito",
    "marg": "margarina",
    "maion": "maionese",
    "shamp": "shampoo",
    "energ": "energetico",
    "grf": "garrafa",
    "lt": "lata",
}

# unidades de medida -> forma canônica (sem espaço, sufixo padronizado)
_UNIT_MAP = {
    "l": "l", "lt": "l", "lts": "l", "litro": "l", "litros": "l", "ltr": "l",
    "ml": "ml", "mls": "ml",
    "g": "g", "gr": "g", "grs": "g", "grama": "g", "gramas": "g",
    "kg": "kg", "kgs": "kg", "quilo": "kg", "kilo": "kg",
    "mg": "mg",
    "un": "", "und": "", "unid": "", "unidade": "", "unidades": "", "u": "",
}

_PACK_WORDS = {"cx", "fd", "pct", "dz", "fardo", "caixa"}  # multiplicidade de embalagem


def strip_accents(text: str) -> str:
    return "".join(
        c for c in unicodedata.normalize("NFKD", text) if not unicodedata.combining(c)
    )


def normalize(text: str) -> str:
    """Normaliza um texto (query ou nome de produto) para o matching."""
    t = strip_accents(str(text).lower())

    # s/acucar -> sem acucar ; c/ gas -> com gas  (antes de remover pontuação)
    t = re.sub(r"\bs/\s*(?=[a-z])", "sem ", t)
    t = re.sub(r"\bc/\s*(?=[a-z])", "com ", t)

    # quantidades de embalagem: c/6, c-12, 12x350ml (mantém o tamanho unitário)
    t = re.sub(r"\bc[/\-]?\d+\b", " ", t)                       # c/6, c12
    t = re.sub(r"\b\d+\s*x\s*(\d+)", r" \1", t)                 # 12x350ml -> 350ml
    t = re.sub(r"\b(cx|fd|pct|dz)\s*\d+\b", " ", t)             # cx12, fd6

    # separa número de unidade colada (350ml -> 350 ml) para canonizar depois
    t = re.sub(r"(\d+)[\.,](\d+)", r"\1.\2", t)                 # 1,5l -> 1.5l
    t = re.sub(r"(\d+(?:\.\d+)?)\s*([a-z]+)", r"\1 \2", t)

    # pontuação -> espaço
    t = re.sub(r"[^a-z0-9.\s]", " ", t)

    tokens = []
    parts = t.split()
    i = 0
    while i < len(parts):
        tok = parts[i]
        # número seguido de unidade -> token canônico colado (2 l -> 2l)
        if re.fullmatch(r"\d+(?:\.\d+)?", tok) and i + 1 < len(parts):
            unit = _UNIT_MAP.get(parts[i + 1])
            if unit is not None:
                num = tok.rstrip("0").rstrip(".") if "." in tok else tok
                if unit:                      # 2 litros -> 2l ; 350 ml -> 350ml
                    tokens.append(num + unit)
                # unidade de contagem (6 und) -> descarta número e unidade
                i += 2
                continue
        if tok in _PACK_WORDS or _UNIT_MAP.get(tok) == "":
            i += 1
            continue
        tok = ABBREVIATIONS.get(tok, tok)
        if tok and tok not in STOPWORDS:
            tokens.append(tok)
        i += 1
    return " ".join(tokens)


def tokenize(text: str) -> list:
    """Normaliza e devolve a lista de tokens (para o BM25)."""
    return normalize(text).split()


exemplos = ["COCA COLA 1L C/6", "MONSTER BRANCO S/ACUCAR C/6 UNID",
            "FINI MINI BANANAS 12X15 GR Und", "CERV HEINEKEN LONG NECK 330ML CX24",
            "Suco de uva orgânico aliança 1 litro", "1,5L AGUA C/ GAS"]
pd.DataFrame({"original": exemplos, "normalizado": [normalize(e) for e in exemplos]})

,original,normalizado
0,COCA COLA 1L C/6,coca cola 1l
1,MONSTER BRANCO S/ACUCAR C/6 UNID,monster branco sem acucar
2,FINI MINI BANANAS 12X15 GR Und,fini mini bananas 15g
3,CERV HEINEKEN LONG NECK 330ML CX24,cerveja heineken long neck 330ml
4,Suco de uva orgânico aliança 1 litro,suco uva organico alianca 1l
5,"1,5L AGUA C/ GAS",1.5l agua gas


## 3. Métricas de avaliação

P@1 (produto correto em 1º), MRR@5 (crédito parcial por posição, 1/rank, 0 se fora do top-5)
e R@5 (correto em alguma das 5 primeiras posições), conforme o enunciado.

In [6]:
"""
Métricas de avaliação do trabalho: P@1, MRR@5 e R@5.

Cada ranker devolve, para cada query, uma lista ordenada com os top-5
product_id mais similares. As métricas comparam essa lista com o matched_id.
"""
import time

K = 5


def compute_metrics(ranked_ids, gold_ids, k=K):
    """ranked_ids: lista de listas (top-k ids por query); gold_ids: lista de ids corretos."""
    assert len(ranked_ids) == len(gold_ids)
    n = len(gold_ids)
    p1 = 0
    rr_sum = 0.0
    r5 = 0
    for ranked, gold in zip(ranked_ids, gold_ids):
        topk = list(ranked[:k])
        if topk and topk[0] == gold:
            p1 += 1
        if gold in topk:
            r5 += 1
            rr_sum += 1.0 / (topk.index(gold) + 1)
        # se não aparece no top-k, 1/rank = 0
    return {"P@1": p1 / n, "MRR@5": rr_sum / n, "R@5": r5 / n, "n": n}


def evaluate_ranker(rank_fn, queries_df, k=K):
    """Aplica rank_fn(text) -> [(product_id, score), ...] a cada query e mede o tempo.

    Devolve (metrics, details, elapsed_seconds). details guarda o top-k por query
    para a análise qualitativa.
    """
    ranked_all, details = [], []
    t0 = time.perf_counter()
    for _, row in queries_df.iterrows():
        top = rank_fn(row["text"])[:k]
        ranked_all.append([pid for pid, _ in top])
        details.append({"text": row["text"], "gold": row["matched_id"], "top": top})
    elapsed = time.perf_counter() - t0
    metrics = compute_metrics(ranked_all, queries_df["matched_id"].tolist(), k)
    metrics["tempo_s"] = round(elapsed, 2)
    return metrics, details, elapsed


def fmt(metrics):
    return (f"P@1={metrics['P@1']:.3f}  MRR@5={metrics['MRR@5']:.3f}  "
            f"R@5={metrics['R@5']:.3f}  (n={metrics['n']}, {metrics.get('tempo_s', '?')}s)")

## 4. Abordagem 1 — NLP Clássico

### 4.1 BM25 (implementação própria, com índice invertido)

In [7]:
"""
Implementação própria do BM25 (Okapi BM25) com índice invertido.

score(q, d) = soma_{t em q} IDF(t) * f(t,d)*(k1+1) / (f(t,d) + k1*(1 - b + b*|d|/avgdl))
IDF(t) = ln( (N - df(t) + 0.5) / (df(t) + 0.5) + 1 )
"""
import math
from collections import Counter, defaultdict

import numpy as np


class BM25:
    def __init__(self, tokenized_docs, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.N = len(tokenized_docs)
        self.doc_len = np.array([len(d) for d in tokenized_docs], dtype=np.float64)
        self.avgdl = self.doc_len.mean() if self.N else 0.0

        # índice invertido: termo -> [(doc_id, freq), ...]
        self.index = defaultdict(list)
        for doc_id, doc in enumerate(tokenized_docs):
            for term, freq in Counter(doc).items():
                self.index[term].append((doc_id, freq))

        self.idf = {
            term: math.log((self.N - len(posts) + 0.5) / (len(posts) + 0.5) + 1.0)
            for term, posts in self.index.items()
        }

    def scores(self, query_tokens):
        """Vetor de scores BM25 da query contra todos os documentos."""
        s = np.zeros(self.N)
        norm = self.k1 * (1.0 - self.b + self.b * self.doc_len / self.avgdl)
        for term in query_tokens:
            posts = self.index.get(term)
            if not posts:
                continue
            idf = self.idf[term]
            for doc_id, freq in posts:
                s[doc_id] += idf * freq * (self.k1 + 1.0) / (freq + norm[doc_id])
        return s

    def topk(self, query_tokens, k=5):
        s = self.scores(query_tokens)
        # ordenação estável: empates resolvidos pela ordem do catálogo (determinístico)
        top = np.argsort(-s, kind="stable")[:k]
        return [(int(i), float(s[i])) for i in top]

### 4.2 Pipelines TF-IDF e BM25

O catálogo é indexado como `product_name` + `brand_name` (quando a marca ainda não aparece
no nome). Em empates de score, a ordenação **estável** garante desempate determinístico
pela ordem do catálogo (reprodutibilidade).

In [8]:
def load_catalog(use_brand=True):
    catalog = pd.read_csv(DADOS / "catalog.csv", dtype=str)
    if use_brand:
        # acrescenta a marca quando ela ainda não aparece no nome do produto
        def doc(row):
            name, brand = row["product_name"], str(row["brand_name"])
            if brand and brand.lower() not in name.lower():
                return f"{name} {brand}"
            return name
        catalog["doc_raw"] = catalog.apply(doc, axis=1)
    else:
        catalog["doc_raw"] = catalog["product_name"]
    catalog["doc"] = catalog["doc_raw"].map(normalize)
    return catalog


def make_tfidf_ranker(catalog, analyzer="word"):
    if analyzer == "word":
        vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
    else:
        vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True)
    doc_matrix = vec.fit_transform(catalog["doc"])
    ids = catalog["product_id"].tolist()
    names = catalog["product_name"].tolist()

    def rank(text, k=5):
        q = vec.transform([normalize(text)])
        sims = cosine_similarity(q, doc_matrix).ravel()
        # ordenação estável: empates são resolvidos pela ordem do catálogo,
        # garantindo resultados determinísticos e reprodutíveis
        top = np.argsort(-sims, kind="stable")[:k]
        return [(ids[i], float(sims[i])) for i in top]

    rank.names = names
    return rank


def make_bm25_ranker(catalog, k1=1.5, b=0.75):
    docs = [d.split() for d in catalog["doc"]]
    bm25 = BM25(docs, k1=k1, b=b)
    ids = catalog["product_id"].tolist()
    names = catalog["product_name"].tolist()

    def rank(text, k=5):
        return [(ids[i], s) for i, s in bm25.topk(tokenize(text), k)]

    rank.names = names
    rank.bm25 = bm25
    return rank

### 4.3 Avaliação na validação (250 queries)

In [9]:
catalogo = load_catalog(use_brand=True)
id2name = dict(zip(catalogo["product_id"], catalogo["product_name"]))

rankers_classicos = {
    "tfidf_word": make_tfidf_ranker(catalogo, analyzer="word"),
    "tfidf_char": make_tfidf_ranker(catalogo, analyzer="char"),
    "bm25":       make_bm25_ranker(catalogo),
}
det_val = {}
for nome, rk in rankers_classicos.items():
    m, det_val[nome], _ = evaluate_ranker(rk, val)
    print(f"{nome:<11} {fmt(m)}")

tfidf_word  P@1=0.952  MRR@5=0.972  R@5=1.000  (n=250, 1.32s)


tfidf_char  P@1=0.972  MRR@5=0.983  R@5=1.000  (n=250, 10.37s)


bm25        P@1=0.964  MRR@5=0.980  R@5=1.000  (n=250, 0.21s)


**Ablações testadas na validação** (decididas aqui, nunca no teste):
indexar sem a marca não alterou nenhuma métrica; BM25 com k₁/b padrão (1,5/0,75) ficou a
0,8 p.p. do TF-IDF-char com custo ~40× menor. **Variante selecionada: TF-IDF de
caracteres (3–5)** — n-gramas de caracteres são robustos a erros ortográficos e abreviações
residuais.

## 5. Abordagem 2 — Deep Learning (embeddings semânticos)

Modelo `paraphrase-multilingual-MiniLM-L12-v2` (Sentence-Transformers), local na CPU,
**sem chave de API**. O catálogo é codificado uma única vez (cache em `cache/`).
Variantes: **densa** (cosseno contra os 14.206 produtos) e **híbrida** (BM25 filtra 50
candidatos → o modelo neural decide o ranking final — estratégia recomendada no enunciado).

In [10]:
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"


def get_model():
    from sentence_transformers import SentenceTransformer
    return SentenceTransformer(MODEL_NAME, device="cpu")


def catalog_embeddings(model, docs):
    """Codifica (com cache em disco) todos os documentos do catálogo."""
    CACHE.mkdir(exist_ok=True)
    key = hashlib.md5(("\n".join(docs) + MODEL_NAME).encode("utf-8")).hexdigest()[:12]
    path = CACHE / f"catalog_emb_{key}.npy"
    if path.exists():
        return np.load(path)
    print(f"Codificando {len(docs)} produtos do catálogo (uma única vez)...")
    t0 = time.perf_counter()
    emb = model.encode(docs, batch_size=256, show_progress_bar=True,
                       normalize_embeddings=True, convert_to_numpy=True)
    print(f"Catálogo codificado em {time.perf_counter() - t0:.1f}s")
    np.save(path, emb)
    return emb


def make_dense_ranker(model, catalog, emb):
    ids = catalog["product_id"].tolist()

    def rank(text, k=5):
        q = model.encode([normalize(text)], normalize_embeddings=True,
                         convert_to_numpy=True)[0]
        sims = emb @ q                      # cosseno (vetores já normalizados)
        top = np.argsort(-sims, kind="stable")[:k]   # desempate determinístico
        return [(ids[i], float(sims[i])) for i in top]

    return rank


def make_hybrid_ranker(model, catalog, emb, n_candidates=50):
    """BM25 filtra n_candidates; o modelo neural decide o ranking final."""
    ids = catalog["product_id"].tolist()
    bm25 = BM25([d.split() for d in catalog["doc"]])

    def rank(text, k=5):
        cand = [i for i, _ in bm25.topk(tokenize(text), n_candidates)]
        q = model.encode([normalize(text)], normalize_embeddings=True,
                         convert_to_numpy=True)[0]
        sims = emb[cand] @ q
        order = np.argsort(-sims, kind="stable")[:k]  # desempate determinístico
        return [(ids[cand[i]], float(sims[i])) for i in order]

    return rank

In [11]:
model = get_model()
emb = catalog_embeddings(model, catalogo["doc"].tolist())   # cache: rápido após a 1ª vez

rankers_deep = {
    "deep_dense":  make_dense_ranker(model, catalogo, emb),
    "deep_hybrid": make_hybrid_ranker(model, catalogo, emb),
}
for nome, rk in rankers_deep.items():
    m, det_val[nome], _ = evaluate_ranker(rk, val)
    print(f"{nome:<11} {fmt(m)}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


deep_dense  P@1=0.920  MRR@5=0.945  R@5=0.984  (n=250, 13.3s)


deep_hybrid P@1=0.924  MRR@5=0.952  R@5=0.992  (n=250, 9.99s)


**Ablação crucial (validação):** codificando o texto **cru** (sem o
pré-processamento da Seção 2), a abordagem neural despenca para P@1 = 0,520 (densa) e
0,588 (híbrida) — queda de ~40 p.p. O modelo semântico não conhece o jargão de pedidos
(`C/6`, `CERV`, `12X350ML`); a normalização é crítica para as duas abordagens.
(Reprodução: `python src/approach2_deep.py --split val --raw-text`.)

**Variante selecionada: híbrida** (melhor R@5 na validação e menor custo por query).

## 6. Avaliação final no conjunto de teste

`queries_test.csv` é usado **uma única vez**, aqui, para as métricas definitivas — nenhuma
decisão de desenvolvimento foi tomada com ele.

In [12]:
todos = {**rankers_classicos, **rankers_deep}
linhas, det_test = [], {}
for nome, rk in todos.items():
    m, det_test[nome], _ = evaluate_ranker(rk, test)
    linhas.append({"variante": nome, **{k: round(v, 3) for k, v in m.items() if k != "n"}})
pd.DataFrame(linhas).set_index("variante")

,P@1,MRR@5,R@5,tempo_s
variante,,,,
tfidf_word,0.932,0.958,0.996,1.76
tfidf_char,0.992,0.996,1.000,13.16
bm25,0.988,0.992,1.000,0.29
deep_dense,0.944,0.962,0.984,7.31
deep_hybrid,0.944,0.962,0.984,7.09


In [13]:
# Tabela comparativa do enunciado (variantes selecionadas na validação)
comp = pd.DataFrame([
    {"Abordagem": "TF-IDF char (clássica)", **{k: round(v,3) for k,v in
        compute_metrics([[p for p,_ in d['top']] for d in det_test['tfidf_char']],
                        test['matched_id'].tolist()).items() if k!='n'},
     "Custo": "Gratuito (CPU)", "Complexidade": "Baixa"},
    {"Abordagem": "BM25 própria (clássica)", **{k: round(v,3) for k,v in
        compute_metrics([[p for p,_ in d['top']] for d in det_test['bm25']],
                        test['matched_id'].tolist()).items() if k!='n'},
     "Custo": "Gratuito (CPU)", "Complexidade": "Baixa"},
    {"Abordagem": "Híbrida BM25+NN (deep)", **{k: round(v,3) for k,v in
        compute_metrics([[p for p,_ in d['top']] for d in det_test['deep_hybrid']],
                        test['matched_id'].tolist()).items() if k!='n'},
     "Custo": "Gratuito (local; modelo 470 MB)", "Complexidade": "Média"},
]).set_index("Abordagem")
comp

,P@1,MRR@5,R@5,Custo,Complexidade
Abordagem,,,,,
TF-IDF char (clássica),0.992,0.996,1.000,Gratuito (CPU),Baixa
BM25 própria (clássica),0.988,0.992,1.000,Gratuito (CPU),Baixa
Híbrida BM25+NN (deep),0.944,0.962,0.984,Gratuito (local; modelo 470 MB),Média


## 7. Análise qualitativa

### 7.1 Acertos, erros e comparação direta (teste)

In [14]:
def tabela_erros(details, nome):
    hits = [d for d in details if d["top"] and d["top"][0][0] == d["gold"]]
    errs = [d for d in details if not d["top"] or d["top"][0][0] != d["gold"]]
    print(f"{nome}: {len(hits)} acertos, {len(errs)} erros (P@1={len(hits)/len(details):.3f})")
    rows = []
    for d in errs:
        ids5 = [p for p, _ in d["top"]]
        rows.append({"query": d["text"],
                     "esperado": id2name.get(d["gold"], "?"),
                     "obtido (top-1)": id2name.get(ids5[0], "—"),
                     "pos. do correto": ids5.index(d["gold"]) + 1 if d["gold"] in ids5 else ">5"})
    return pd.DataFrame(rows), {d["text"] for d in errs}

df_err_c, errs_c = tabela_erros(det_test["tfidf_char"], "Abordagem 1 (tfidf_char)")
display(df_err_c)
df_err_d, errs_d = tabela_erros(det_test["deep_hybrid"], "Abordagem 2 (deep_hybrid)")
display(df_err_d)
print(f"Só o clássico errou (deep acertou): {len(errs_c - errs_d)}")
print(f"Só o deep errou (clássico acertou): {len(errs_d - errs_c)}")
print(f"Erros em comum: {len(errs_c & errs_d)}")

Abordagem 1 (tfidf_char): 248 acertos, 2 erros (P@1=0.992)


,query,esperado,obtido (top-1),pos. do correto
0,Fralda Pampers Confort Sec G 128 Unidades,Fralda pampers confort sec g 128 unidades,Fralda pampers confort sec g 60 unidades,2
1,Espumante Casa Valduga Arte Tradicional Brut 750ml,Espumante brut branco arte tradicional casa valduga 750ml,Espumante brut rosé arte tradicional casa valduga 750ml,2


Abordagem 2 (deep_hybrid): 236 acertos, 14 erros (P@1=0.944)


,query,esperado,obtido (top-1),pos. do correto
0,Pipoca para Microondas Manteiga YOKI 100g,Pipoca para micro-ondas manteiga yoki 100g,Saco hermético para freezer e microondas 31x27cm conserv 8 unidades,>5
1,Fralda Pampers Confort Sec G 128 Unidades,Fralda pampers confort sec g 128 unidades,Fralda pampers confort sec g 60 unidades,2
2,Vinho Argentino Tinto Cordero con Piel de Lobo 750ml,Cordero con piel de lobo tinto de tintas argentino vinho tinto 750ml,Cordero con piel de lobo malbec argentino vinho tinto 750ml,>5
3,Espumante Casa Valduga Arte Tradicional Brut 750ml,Espumante brut branco arte tradicional casa valduga 750ml,Espumante brut rosé arte tradicional casa valduga 750ml,2
4,Biscoito Recheado Chocolícia 132g,Biscoito recheado chocolate chocolícia 132g,Biscoito recheado tortinha de limão trakinas 126g,>5
5,Creme Dental Antitártaro Colgate Total 12 Caixa 180g,Creme dental colgate total 12 anti-tártaro 180g,Creme dental colgate total 12 gengiva reforçada 180g,2
6,Refresco em Pó Laranja Docinha Tang Pacote 18g,Refresco em pó laranja docinha tang 18g,Refresco em pó laranja tang 18g,2
7,Queijo Maasdam Président 160g,Queijo tipo maasdam président 160g,Queijo mussarela fatiado président 150g,3
8,Refresco em Pó Maracujá Tang Pacote 18g,Refresco em pó maracujá tang 18g,Refresco em pó manga tang 18g,4
9,Pipoca para Microondas Sabor Natural YOKI com Sal 100g,Pipoca para micro-ondas natural com sal yoki 100g,Salgadinho tostitos sabor toque de sal marinho 110g,>5


Só o clássico errou (deep acertou): 0
Só o deep errou (clássico acertou): 12
Erros em comum: 2


**Leitura dos erros.** Os 2 erros do clássico são casos-limite: a fralda
*128 unidades* (nosso pré-processamento remove "\<n\> unidades" como multiplicidade de
embalagem — mas em fraldas a contagem é o tamanho do produto) e o espumante *brut* sem cor
especificada (rosé × branco — genuinamente ambíguo). Os erros do deep concentram-se em
**variante/sabor errado** (maracujá→manga Tang), **tamanho/marca** (Salsicha Ceratti→
Perdigão) e **categoria arrastada pela semântica** (pipoca de microondas→saco para
microondas).

### 7.2 Casos ambíguos (top-2 com scores quase empatados)

In [15]:
rows = []
for d in det_test["tfidf_char"]:
    if len(d["top"]) >= 2 and d["top"][0][1] - d["top"][1][1] < 0.01:
        rows.append({"query": d["text"],
                     "top-1": id2name.get(d["top"][0][0], "?"),
                     "top-2": id2name.get(d["top"][1][0], "?"),
                     "Δ score": round(d["top"][0][1] - d["top"][1][1], 4)})
pd.DataFrame(rows)

,query,top-1,top-2,Δ score
0,Refrigerante Laranja Fanta Lata 350ml,Refrigerante fanta laranja lata 350ml,Refrigerante fanta laranja lata 350ml 12 un,0.0000
1,Fralda Pampers Confort Sec G 128 Unidades,Fralda pampers confort sec g 60 unidades,Fralda pampers confort sec g 128 unidades,0.0000
2,Leite UHT Semidesnatado Parmalat Garrafa 1 Litro,Leite uht semidesnatado parmalat garrafa 1 litro,Leite uht semidesnatado parmalat garrafa 1 litro 6 un,0.0000
3,Espumante Casa Valduga Arte Tradicional Brut 750ml,Espumante brut rosé arte tradicional casa valduga 750ml,Espumante brut branco arte tradicional casa valduga 750ml,0.0009
4,CORONA CERO SUNBREW LONG NECK 330ML - C/24,Cerveja corona cero sunbrew long neck 330ml,Cerveja corona cero sunbrew long neck 330ml 6 un,0.0000


Os empates são versões **unitária × multipack** ("…lata 350ml" ×
"…lata 350ml 12 un") e variantes não especificadas na query. Com o desempate determinístico
pela ordem do catálogo, a versão unitária (que aparece primeiro) é preferida.

### 7.3 Casos NO_MATCH

Queries reais de `queries.csv` cujas marcas **não existem** no catálogo. Nenhuma abordagem
tem rejeição nativa — sempre devolvem algo; a defesa é um **limiar sobre o score**.

In [16]:
NO_MATCH_QUERIES = [
    "MORMAII LATA BRANCO C/6",
    "Limpador para Casa Perfumado QUALITÁ Lavanda 2 litros",
    "Farinha de Chia Taeq Pouch 150g",
    "Rabo de Bovino Congelado SWIFT 2Kg",
    "CHICLETE BUZZY ROSA MORANGO",
]
CONTROLE = ["Café Solúvel Granulado Forte Nescafé Tradição Sachê 40g",
            "FANTA LARANJA 2L C/6"]

rows = []
for grupo, qs in [("NO_MATCH", NO_MATCH_QUERIES), ("controle (existe)", CONTROLE)]:
    for q in qs:
        for nome in ("tfidf_char", "deep_hybrid"):
            pid, sc = todos[nome](q)[0]
            rows.append({"grupo": grupo, "query": q, "abordagem": nome,
                         "top-1": id2name.get(pid, "?"), "score": round(sc, 3)})
pd.DataFrame(rows)

,grupo,query,abordagem,top-1,score
0,NO_MATCH,MORMAII LATA BRANCO C/6,tfidf_char,Pão de ló branco rio branco 300g,0.407
1,NO_MATCH,MORMAII LATA BRANCO C/6,deep_hybrid,Pão de ló branco rio branco 300g,0.447
2,NO_MATCH,Limpador para Casa Perfumado QUALITÁ Lavanda 2 litros,tfidf_char,Limpador veja perfumes lavanda da frança 2 litros,0.584
3,NO_MATCH,Limpador para Casa Perfumado QUALITÁ Lavanda 2 litros,deep_hybrid,Limpador casa & perfume agradable 1 litro,0.902
4,NO_MATCH,Farinha de Chia Taeq Pouch 150g,tfidf_char,Farinha de chia e linhaça chiaça orgânico ecobio 250g,0.384
5,NO_MATCH,Farinha de Chia Taeq Pouch 150g,deep_hybrid,Iogurte de morango parmalat fit pouch 100g,0.681
6,NO_MATCH,Rabo de Bovino Congelado SWIFT 2Kg,tfidf_char,Assado de tira bovino congelado las piedras,0.366
7,NO_MATCH,Rabo de Bovino Congelado SWIFT 2Kg,deep_hybrid,Assado de tira bovino congelado gran selezione,0.833
8,NO_MATCH,CHICLETE BUZZY ROSA MORANGO,tfidf_char,Pirulito flopito chiclé tutti-frutti florestal 600g,0.239
9,NO_MATCH,CHICLETE BUZZY ROSA MORANGO,deep_hybrid,Chinelo ipanema glitter rosa 37/38,0.649


**Conclusões sobre NO_MATCH:** no TF-IDF-char os scores de NO_MATCH ficam
em 0,24–0,68, bem abaixo dos matches corretos descritivos (p5 = 0,880 na validação) — um
limiar funciona. Na neural, os scores continuam altos mesmo sem o produto existir (até 0,90
para o limpador Qualitá), pois sempre há substituto semanticamente próximo — limiar pouco
confiável. **O limiar depende do estilo da query**: no estilo abreviado de `queries.csv`, um
match correto pode pontuar ~0,73 ("FANTA LARANJA 2L C/6") e NO_MATCHes ficam em ~0,41–0,43;
≈0,55 separa razoavelmente os regimes nesse estilo.

## 8. Preenchimento de `queries.csv`

A descrição dos dados diz que `matched_id` é "a ser preenchido pelo grupo". Aplicamos a
variante vencedora (TF-IDF-char) em lote às 16.441 queries, gravando também o `score` do
top-1 para permitir o filtro de possíveis NO_MATCH. O arquivo original em `Dados/` **não é
modificado**.

In [17]:
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True)
doc_matrix = l2_normalize(vec.fit_transform(catalogo["doc"]))
ids_arr = np.array(catalogo["product_id"])

t0 = time.perf_counter()
q_norm = [normalize(t) for t in queries["text"]]
matched, scores = [], []
for ini in range(0, len(queries), 2000):
    q_m = l2_normalize(vec.transform(q_norm[ini:ini + 2000]))
    sims = (q_m @ doc_matrix.T).toarray()
    best = sims.argmax(axis=1)
    matched.extend(ids_arr[best])
    scores.extend(sims[np.arange(len(best)), best])

preenchido = pd.DataFrame({"text": queries["text"], "matched_id": matched,
                           "score": np.round(scores, 4)})
preenchido.to_csv(RESULTS / "queries_preenchido.csv", index=False, encoding="utf-8")
print(f"{len(preenchido)} queries preenchidas em {time.perf_counter()-t0:.1f}s")
for limiar in (0.45, 0.55, 0.65):
    n = (preenchido['score'] < limiar).sum()
    print(f"score < {limiar:.2f} (possível NO_MATCH): {n} ({n/len(preenchido):.1%})")
preenchido.head(8)

16441 queries preenchidas em 11.7s
score < 0.45 (possível NO_MATCH): 3792 (23.1%)
score < 0.55 (possível NO_MATCH): 6751 (41.1%)
score < 0.65 (possível NO_MATCH): 9669 (58.8%)


,text,matched_id,score
0,FINI MINI BANANAS 12X15 GR Und,07898279790394,0.6173
1,MORMAII LATA BRANCO C/6,07896277901422,0.4075
2,FANTA LARANJA 2L C/6,07894900031515,0.7306
3,SKOL LATA 350ml C/12,07891149105571,0.4267
4,MATUTA VERDE UMBURANA 1L UND,07897877000249,0.5099
5,MONSTER BRANCO S/ACUCAR C/6 UNID,00070847022206,0.4907
6,PITU LATA 350ml C/12,07894900050011,0.3394
7,CACHACA COBICADA UMBURANA 250ml C/6,07897877000249,0.6832


## 9. Conclusões

| Abordagem (teste, 250 q) | P@1 | MRR@5 | R@5 | Tempo | Custo | Complexidade |
|---|---|---|---|---|---|---|
| **TF-IDF caracteres** | **0,992** | **0,996** | **1,000** | ~12 s | Gratuito | Baixa |
| BM25 (própria) | 0,988 | 0,992 | 1,000 | ~0,2 s | Gratuito | Baixa |
| Híbrida BM25 + MiniLM | 0,944 | 0,962 | 0,984 | ~7 s (+106 s de indexação única) | Gratuito (local) | Média |

1. **Em catálogo de vocabulário controlado, o clássico bem pré-processado vence**: TF-IDF-char
   0,992 e BM25 0,988 (a ~1 ms/query) contra 0,944 da neural.
2. **O pré-processamento domina o resultado**: sem ele a neural perde ~40 p.p. de P@1 —
   é a tradução do jargão (`C/6`, `CERV`) para o vocabulário do catálogo que resolve a tarefa.
3. **Os erros restantes são de granularidade**, não de entendimento: multipack, sabor, cor,
   contagem — pedem extração estruturada de atributos, não modelos maiores.
4. **Melhorias futuras**: fine-tuning do bi-encoder (triplet loss com pares minerados de
   `queries.csv`), re-rank com cross-encoder ou LLM (zero/few-shot nos top-10 do BM25),
   limiar de NO_MATCH calibrado por estilo de query, fusão de scores BM25+embedding.